In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration


In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
val_data.head()

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [5]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [6]:
train_data.sample(10)

,id,dialogue,summary
13980,13729522,Kevin: You know what's been bothering me?\r\nP...,Trump and Kim Dzong Un met. There are no updat...
13389,13730781,Kaz: <file_photo>\r\nMatt: LOL \r\nKaz: Ashi o...,Kaz and Matt's cat thinks that kitchen sinks m...
9877,13821819,Tanya: We're leaving Ireland tomorrow :(\r\nLe...,Tanya is leaving Ireland tomorrow.
13209,13862477,Chaima: heeeey! can you help me choose a dress...,Meriem is helping Chaima choose a dress for Ch...
843,13680465,"Alice: Good morning, are we meeting on Saturda...",On Saturday Matthew will be available after 6 ...
10488,13716697,Sara: Have you been in the dressing up box aga...,"Sara, Ceri, Marsha, Mary, Clare and Elizabeth ..."
11982,13730351,Debbie: You'll never guess what.\r\nRuth: What...,Abigail is pregnant. The church may force her ...
4667,13729616,Jessie: <file_picture>\r\nJessie: <file_pictur...,Jessie has sent Logan some suggestions of the ...
7571,13820840,"Sarah: Hey guys I’m still at Moncloa, waiting ...",Sarah is at Moncloa waiting for the bus. She s...
225,13731461,Emma: We are going beach would you like to joi...,"Sharol is going to go to the beach with Emma, ..."


In [7]:
train_data.shape

(14732, 3)

In [8]:
val_data.shape

(818, 3)

In [9]:
#Random Sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [10]:
train_data.shape

(4000, 3)

## Data Preprocessing

In [11]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) #lines"
    text = re.sub(r"\s+", " ", text) # Spaces
    text = re.sub(r"<.*?>", "", text) # HTML tags
    text = text.strip().lower()
    return text





In [12]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

In [13]:
val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [14]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:  claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

### Tokenize

In [15]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")


In [16]:
# Raw Data => Tokenized Inputs for fine-tuning the model
def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", truncation=True, max_length=512)
    targets = tokenizer(data["summary"], padding="max_length", truncation=True, max_length=150)

    inputs["labels"] = targets["input_ids"] # token ids => add to inputs as labels for training

    return inputs

In [17]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [18]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [19]:
# input_ids - dialogue => token ids
# 1 => EOS, 0 => padding
# attention_mask
# labels -  summary => token ids - target for training
# 1 -> valid token, 0 -> padding
len(train_dataset[0]["input_ids"])



512

In [20]:
len(train_dataset[0]["labels"])

150

In [21]:
len(train_dataset[0]["attention_mask"])

512

In [22]:
type(train_dataset[0])

transformers.tokenization_utils_base.BatchEncoding

In [23]:
type(train_dataset)

list

In [24]:
type(val_dataset)

list

## Working With Our Model

In [25]:
# NLP => Generation Task => T5 Model => Conditional Generation => T5ForConditionalGeneration => Based on the input, generate the output

model = T5ForConditionalGeneration.from_pretrained("t5-small")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [26]:
# Fine-tuning the model
import torch 

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device: ", device)
model.to(device)

Device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [27]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.14.0+cu130
CUDA available: True
CUDA version: 13.0
GPU count: 1
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [36]:
training_args = TrainingArguments(
    output_dir="./results",

    # Training
    num_train_epochs=10,
    learning_rate=5e-5,
    weight_decay=0.01,

    # Batch
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,

    # Evaluation & checkpoints
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    # Learning-rate warmup
    warmup_steps=500,

    # GPU optimization
    fp16=True,

    # Logging
    logging_steps=50,

    # Restore best checkpoint
    load_best_model_at_end=True,

    
)

In [37]:
from pathlib import Path

print("Current working directory:", Path.cwd())
print("Results directory:", Path("./results").resolve())

Current working directory: d:\AIML\projects\brief-sync
Results directory: D:\AIML\projects\brief-sync\results


In [38]:
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
